# NB00 — Setup & Verification

        **Run this first, on every account. ~15 minutes. Nothing here is wasted
        if it passes, and everything downstream is wasted if it doesn't.**


> **New here?** Read `05_PLAIN_ENGLISH_GUIDE.md` first — it explains what this
> project is measuring and why, without jargon. This notebook assumes you have.


        ## What this notebook is for

        Every check below corresponds to a specific failure that would otherwise
        surface hours into a real run:

        | Check | What it saves you from |
        |---|---|
        | HuggingFace reachable **and writable** | Nine hours of training with nowhere to put it |
        | CIFAR-100 found | Silently falling back to a slow download |
        | All 15 model types build and train | A Vision Transformer whose internals don't fit our measurement code |
        | FLOPs measured correctly | A compute scale that doesn't actually reach 100% |
        | **Kill-and-resume works** | See below — this is the important one |
        | Worker splitting is balanced | One account working 33 hours while another idles |

        ## Why the resume test matters more than it looks

        We kill a training run halfway and restart it, then check the result is
        *identical* to an uninterrupted run.

        The subtle failure it catches: a resume that reloads the model but not
        the random-number state. Training looks fine. But the resumed run now
        sees images in a different order than it would have. That breaks the
        meaning of "same model, different random seed" — which is exactly the
        comparison Q1 relies on, and Q1 is the denominator of every number in
        the paper.

        So we don't assume it works. We test it.

        ## Before you press Run

        1. `HF_TOKEN` in **Add-ons → Secrets**, with **WRITE** permission
        2. Internet **ON**
        3. Accelerator: **GPU T4 × 2**
        4. Dataset attached: `shanmuk4622/dataset-cifar100-python`

        Everything is saved to **`Shanmuk4622/msc-cifar100`** — one repository, one folder per run.

In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# This writes two Python files into the session and imports them. Nothing here
# touches the GPU or the network beyond installing three small packages.
#
#   msc_lib   b93fedb20fbf   the pipeline: HuggingFace sync, model zoo,
#                              measurement, training, the method
#   msc_core  2571e11b4f7e   the reference maths: the MSC definition and
#                              every statistic in the paper
#
# Both are generated from KD/src by build_notebooks.py. Editing them HERE does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle images already ship torch, pandas and sklearn. These three vary by
# image version, so we check rather than assume.
#   pyarrow  writes the per-image measurement tables (Parquet)
#   pynvml   reads GPU power/temperature/utilisation directly
#   fvcore   counts FLOPs, which is how compute cost is defined
for _pkg in ('pyarrow', 'pynvml', 'fvcore', 'psutil'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg,
                        '--break-system-packages'], check=False)

_LIB = (
    '...',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]

import msc_lib as msc
import msc_core

print(f'[BOOT] msc_lib v{msc.__version__} ready  (torch available: {msc._TORCH_OK})')
print(f'[BOOT] artifact space: {msc.WORK_ROOT}   scratch space: {msc.SCRATCH_ROOT}')

## Step 1 — Start a session

In [ ]:
# === Who am I? =============================================================
#
# ACCOUNT   labels this Kaggle account in the run log. Two accounts calling
#           themselves the same thing makes the log useless.
# WORKER_ID splits the work. Every account runs THE SAME notebook; the only
#           thing that differs is this number. Account 1 -> 0, account 2 -> 1,
#           and so on. Each account then works out, by pure arithmetic, which
#           jobs belong to it -- no communication needed, no chance of two
#           accounts training the same model, no chance of a job being missed.
#
# DEFAULT IS 1: this account does everything in this notebook. That is the
# simplest thing that works. Change it only when you actually have several
# accounts running at once.
ACCOUNT     = 'acct1'      # <<< CHANGE ME
NUM_WORKERS = 1          # <<< how many accounts you are running in parallel
WORKER_ID   = 0            # <<< CHANGE ME: 0, 1, 2, ... up to NUM_WORKERS-1

sess = msc.Session(account=ACCOUNT, phase='test', dataset='cifar100',
                   worker_id=WORKER_ID, num_workers=NUM_WORKERS,
                   shard_mode='cost',        # balance GPU-hours, not job counts
                   enable_hf=True,
                   session_limit_h=8.5,      # push + pause before Kaggle kills us
                   commits_per_hour_limit=20,# 6 accounts x 20 = 120 < HF's 128/hr
                   batch_interval_sec=1800)  # the 30-minute push policy

## Step 2 — Get the dataset

In [ ]:
# === Get CIFAR-100 =========================================================
# Looks in this order: attached Kaggle dataset (instant) -> earlier extraction
# -> Kaggle CLI download -> torchvision as a last resort.
# Everything lands in /kaggle/temp (~1 TB scratch), never in /kaggle/working
# (20 GB, and that is the space your results need).
DATA_ROOT = sess.prepare_data()
print('dataset:', DATA_ROOT)

## Step 3 — Check everything

Builds all 15 architectures, runs data through them, attaches an early-exit
head, and backpropagates. `quick=False` also measures each one's FLOPs
table and checks the compute scale is sane (increasing, and ending at
exactly 100%).

This cell `assert`s. If it stops, fix what it names before continuing.

In [ ]:
report = msc.preflight(sess, archs=list(msc.ZOO.keys()), quick=False)

p = sess.data_dir / 'analysis' / f'preflight_{sess.account}.json'
msc.atomic_write_json(p, report)
if sess.hub.enabled:
    sess.hub.hub.enqueue(p, f'analysis/preflight_{sess.account}.json')

assert report['all_passed'], 'Preflight failed. Fix the FAILs above before training.'

## Step 4 — Measure the compute cost of every setting

This builds the table that defines what "50% compute" *means* for each
architecture.

`rho` is the fraction-of-full-cost for each setting. It's the number
that lets us compare a ResNet to a Transformer at all — without it,
"needed 12 layers" and "needed 6 blocks" aren't comparable quantities.

Two columns worth understanding:

**`K`** is how many distinct depth settings that architecture supports.
Usually 5. But `resnet8x4` has only 3 blocks in total, so it physically
cannot have five distinct early-exit points — it gets K=3. That's
correct, not a bug: MSC is a cost *fraction*, not an exit index, so
architectures may carry different K and still be comparable. (Forcing 5
would produce duplicate budgets, which makes "the smallest sufficient
one" ill-defined and would crash the measurement later.)

**`res_native`** is whether the architecture can run on a smaller image
at all. MLP-Mixer cannot — its token-mixing layer is a linear map whose
input size *is* the patch count — so it shows `False` and its resolution
axis uses the shrink-then-restore proxy. Documented limitation, not a
failure.

Measured once and frozen. A cost table that drifts between sessions
would make measurements from different sessions incomparable.

In [ ]:
import pandas as pd
rows = []
for arch in msc.ZOO:
    b = sess.budgets(arch, 100)
    d, r, q = b['axes']['depth'], b['axes']['resolution'], b['axes']['precision']
    rows.append({'arch': arch, 'family': msc.ZOO[arch]['family'],
                 'params_M': round(b['params'] / 1e6, 2),
                 'full_GFLOPs': round(b['full_flops'] / 1e9, 3),
                 'K': d['K'], 'blocks': d['n_blocks'],
                 'depth_rho': [round(x, 3) for x in d['rho']],
                 'res_native': r['native_supported'],
                 'res_rho': [round(x, 3) for x in r['rho']],
                 'prec_rho': [round(x, 3) for x in q['rho']]})
bt = pd.DataFrame(rows)
display(bt)

# Three invariants the measurement code depends on.
for _r in rows:
    rho = _r['depth_rho']
    assert abs(rho[-1] - 1.0) <= 0.02, \
        f"{_r['arch']}: deepest exit does not cost the full model ({rho})"
    assert all(rho[i] < rho[i + 1] for i in range(len(rho) - 1)), \
        f"{_r['arch']}: depth costs not strictly ascending ({rho})"
    assert len(set(rho)) == len(rho), \
        f"{_r['arch']}: DUPLICATE depth budgets ({rho}) -- the oracle cannot use these"
print('\nAll depth cost curves are strictly ascending and reach 1.0.')

odd = bt[bt.K < 5]
if len(odd):
    print(f'\n{len(odd)} architecture(s) carry fewer than 5 depth settings '
          f'(too few blocks). Expected and handled:')
    display(odd[['arch', 'blocks', 'K', 'depth_rho']])
nonat = bt[~bt.res_native]
if len(nonat):
    print(f'\n{len(nonat)} architecture(s) cannot run at non-32px input; '
          f'their resolution axis uses the proxy:')
    display(nonat[['arch', 'res_native']])
print('\nprofiler used:', sess.budgets('resnet20')['profiler'])

## Step 5 — The kill-and-resume test

Two runs of the **same config**:

1. **Reference** — 4 epochs straight through.
2. **Interrupted** — killed by a *real* `KeyboardInterrupt` after epoch 2,
   exercising the actual emergency-flush and paused-state path, then
   resumed in a fresh call.

Then it compares them. Passing requires:

- the interrupt actually fired
- the resumed run reaches all 4 epochs
- no duplicated epoch rows in the log
- **per-epoch training loss after the seam matches the reference within 5%**

That last one is the point. It's where a lost random-number state shows
up: if the image order diverges on resume, the post-seam losses drift
even though nothing looks broken. A resumed run that isn't equivalent to
an uninterrupted one makes "same model, different seed" meaningless —
and that comparison is the denominator of every number in the paper.

You'll see the two loss curves printed side by side so you can check it
yourself rather than trusting a boolean.

In [ ]:
res = msc.resume_acceptance_test(sess, arch='resnet20', epochs=4, kill_at=2)
assert res['ok'], (f'RESUME TEST FAILED: {res}\n'
                   'Do not start the atlas until this passes — every long run '
                   'depends on it.')
res

## Step 6 — Check the work splits evenly

Shows how the 45 Phase-1 runs would be divided among N accounts.

The number to look at is **imbalance**. The phase isn't finished until
the *slowest* worker finishes, so a 3× imbalance means the phase takes
3× longer than it needs to.

Simple hashing (what a straightforward implementation does) gives about
4.9× here, because 45 jobs of very unequal size don't hash evenly. Our
cost-balanced scheduler gets it to about 1.02×.

In [ ]:
run_ids = [c['run_id'] for c in msc.phase1_configs()]
print(f'Phase 1 is {len(run_ids)} training runs.\n')
for mode in ('hash', 'balanced', 'cost'):
    print(f'--- mode = {mode} ---')
    display(msc.shard_report(run_ids, NUM_WORKERS if NUM_WORKERS > 1 else 6, mode=mode))

## Step 7 — Prove we can actually write to HuggingFace

Uploads a small file, then **re-lists the repository to confirm it
arrived**. An upload that didn't raise an error is not evidence that
anything was written — a read-only token fails exactly this way.

In [ ]:
probe = sess.data_dir / 'analysis' / f'smoketest_{sess.account}.json'
msc.atomic_write_json(probe, {'account': sess.account, 'worker': WORKER_ID,
                              'utc': msc.now_iso(), 'env': msc.environment_report()})
if sess.hub.enabled:
    sess.hub.hub.enqueue(probe, f'analysis/smoketest_{sess.account}.json')
    sess.hub.flush(timeout=300)
    files = sess.hub.hub.list_repo_files()
    ok = f'analysis/smoketest_{sess.account}.json' in files
    print(f'probe arrived on HF: {ok}   ({len(files)} files in the data repo)')
    assert ok, 'Upload did not arrive. Your HF_TOKEN probably lacks WRITE scope.'
    print(f'{len(files)} files in {sess.hub.repo_id}')
else:
    print('HF is disabled -- fix this before running anything else.')
sess.hub.print_stats()

## Step 8 — Finish

In [ ]:
# === Push everything and stop ==============================================
# Blocks until HuggingFace confirms. Safe to re-run.
sess.finish()

---
### Everything passed?

Go to **NB01**. Do not skip Phase 0 — it's 12 GPU-hours and it decides
whether the remaining ~1,180 are worth spending.